# Model B, B1 — gaps to the human targets (Oct 30, 2026) — EXPLORATORY, descriptive

**Spec (as given):**

> Descriptive gap table from p24_checks.csv (no simulation): for every statistic and cell, the human value and the gap (sim - human)/CI half-width for L1, final and B1, plus the change B1 - L1 in half-widths. Save it as results/tables/pB1_gaps.csv and print it.
> Also, from the B1 replicates if available (else skip): the median crossing time in smooth vs. stop-and-go crossings, B1 vs. humans, per interface.

**Choices.**
- Every row of `p24_checks.csv` (the 53 check statistics plus the L1b-only rows; `in_check_list` says which).
- Half-width = (ci_hi - ci_lo) / 2 of the human CI used by the checks: the bootstrap CI, or the rule that replaced a zero-width CI (`ci_rule`).
- gap = (sim mean - human) / half-width. A gap in [-1, 1] is about a pass; CIs are not symmetric, so the verdicts in `p24_checks.csv` decide pass / miss, not this table.
- `B1_minus_L1` = gap B1 - gap L1. It is signed: whether B1 is closer to the human value is |gap B1| < |gap L1|.
- **Crossing times, smooth vs. stop-and-go: skipped.** `p24_reps_B1.parquet` holds only the per-replicate check statistics (one median crossing time per cell), not crossings, so the split needs a new simulation.

In [1]:
import numpy as np
import pandas as pd

from slider import config

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

checks = pd.read_csv(config.P24_CHECKS)
key = ["target", "stat", "topology", "dataset"]
human = checks[checks["preset"] == "L1"].set_index(key)[["in_check_list", "human", "ci_lo", "ci_hi", "ci_rule"]]
human["half_width"] = (human["ci_hi"] - human["ci_lo"]) / 2
gaps = human.drop(columns=["ci_lo", "ci_hi"])
for p in ["L1", "final", "B1"]:
    sim = checks[checks["preset"] == p].set_index(key)
    assert np.allclose(sim["human"], human["human"])                 # the same human targets for every preset
    gaps[f"gap_{p}"] = (sim["sim_mean"] - human["human"]) / human["half_width"]
    gaps[f"verdict_{p}"] = sim["verdict"]
gaps["B1_minus_L1"] = gaps["gap_B1"] - gaps["gap_L1"]
gaps = gaps.reset_index()
gaps.to_csv(config.PB1_GAPS, index=False)
print(gaps.drop(columns=["ci_rule"]).round(2).to_string())

    target                            stat   topology dataset  in_check_list   human  half_width  gap_L1 verdict_L1  gap_final verdict_final  gap_B1 verdict_B1  B1_minus_L1
0        1                success_rate_pct       Full      CC           True   97.22        4.55   -8.74       miss     -11.10          miss   -8.79       miss        -0.06
1        1                success_rate_pct       Full      CL           True   82.54       11.11   -3.80       miss      -4.40          miss   -3.96       miss        -0.16
2        1                success_rate_pct  Hierarchy      CC           True   92.50        7.89   -1.15       miss      -1.78          miss   -1.17       miss        -0.03
3        1                success_rate_pct  Hierarchy      CL           True   95.08        5.26   -6.08       miss      -6.86          miss   -6.18       miss        -0.09
4        1                success_rate_pct  Hierarchy      EL           True   94.12        5.88   -1.25       miss      -2.16         

In [2]:
c = gaps[gaps["in_check_list"]]
closer = (c["gap_B1"].abs() < c["gap_L1"].abs())
print(f"53 check statistics: B1 closer than L1 in {closer.sum()}, farther in {(~closer).sum()}; "
      f"median |gap| L1 {c['gap_L1'].abs().median():.2f}, final {c['gap_final'].abs().median():.2f}, "
      f"B1 {c['gap_B1'].abs().median():.2f}; passes L1 {(c['verdict_L1'] == 'pass').sum()}, "
      f"final {(c['verdict_final'] == 'pass').sum()}, B1 {(c['verdict_B1'] == 'pass').sum()}")
c.assign(abs_change=c["gap_B1"].abs() - c["gap_L1"].abs()).groupby(["target", "stat"]).agg(
    cells=("abs_change", "size"), B1_closer=("abs_change", lambda x: (x < 0).sum()),
    median_B1_minus_L1=("B1_minus_L1", "median")).round(2)

53 check statistics: B1 closer than L1 in 34, farther in 19; median |gap| L1 1.80, final 1.86, B1 1.76; passes L1 17, final 16, B1 17


,,cells,B1_closer,median_B1_minus_L1
target,stat,,,
1,success_rate_pct,6,2,-0.04
2,consensus_time_km_median_s,6,2,0.07
3,fraught_pct_mean,3,2,0.00
4,committed_pct_mean,6,2,0.01
5,breaker_excess,2,2,0.04
6,end_sitting_pct,6,2,-0.00
7,relapse_pct,6,6,-0.13
8,reversal_rate,6,6,-0.09
9,f_rev_median,6,5,0.03


**How to read it.** Each row is one check statistic in one cell. `human` is the target; `gap_*` is how far each preset's mean over 200 replicates sits from it, in human-CI half-widths (positive = the model is above the humans). `B1_minus_L1` is B1's gap minus L1's. The summary counts the check statistics where B1 is closer to the humans than L1. Descriptive only: B1 is EXPLORATORY and decides nothing.